# AN7914: Data Analytics and Modelling
## Determinants of Interest Rates in the Consumer Credit Market
**University of Winchester | Dr Sakib Anwar | 2025/26**

This notebook contains all Python code for the summative assignment.
Run cells **top to bottom**. Ensure `loans_dataset.csv` is in the same folder.

**Figures saved:** `fig1_hist_interest_rate.png` through `fig7_box_homeownership.png`

---
## Imports

In [ ]:
# Import all required libraries
import pandas as pd                    # Data loading and manipulation
import numpy as np                     # Numerical operations
import matplotlib.pyplot as plt        # Plotting
import seaborn as sns                  # Enhanced visualisations (imported for style)
from scipy import stats                # Linear regression for OLS overlay on scatterplot
import statsmodels.api as sm           # OLS regression — used for Model 5
import statsmodels.formula.api as smf  # Formula-based OLS — used for Models 1–4
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('All libraries loaded successfully')

---
# Part A: Data Preparation

### 1.1(a) Load Dataset and Subset Variables

In [ ]:
# Load the full raw dataset
df_raw = pd.read_csv('loans_dataset.csv')
print(f'Original dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')

# Select only the 15 required variables as specified in the assignment
cols_to_keep = [
    'interest_rate', 'verified_income', 'debt_to_income',
    'total_credit_utilized', 'total_credit_limit', 'public_record_bankrupt',
    'loan_purpose', 'term', 'inquiries_last_12m',
    'issue_month', 'annual_income', 'loan_amount',
    'grade', 'emp_length', 'homeownership'
]

df_clean = df_raw[cols_to_keep].copy()
print(f'Cleaned dataset: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns')

### 1.1(b) Rename Column

In [ ]:
# Rename inquiries_last_12m to credit_checks for clarity
df_clean.rename(columns={'inquiries_last_12m': 'credit_checks'}, inplace=True)
print('Renamed: inquiries_last_12m  -->  credit_checks')
print('All columns:', list(df_clean.columns))

### 1.1(c) Documentation: Row Counts and Summary Statistics

In [ ]:
# Compare row counts: original vs cleaned
print(f'Original dataset rows : {df_raw.shape[0]}')
print(f'Cleaned dataset rows  : {df_clean.shape[0]}')
print(f'Rows dropped          : {df_raw.shape[0] - df_clean.shape[0]}')

# Summary statistics for all 8 numerical variables
# NOTE: emp_length is categorical (e.g. '1 year', '10+ years') so excluded
numeric_vars = [
    'interest_rate', 'debt_to_income', 'total_credit_utilized',
    'total_credit_limit', 'public_record_bankrupt',
    'credit_checks', 'annual_income', 'loan_amount'
]

summary = df_clean[numeric_vars].describe().round(2)
print('\n--- Summary Statistics (All Numerical Variables) ---')
print(summary.to_string())

---
# Part B: Exploratory Data Analysis

### 2.1 Descriptive Statistics — Continuous Variables

In [ ]:
# Mean, median, std, min, max for key continuous variables
key_vars = ['interest_rate', 'annual_income', 'debt_to_income', 'loan_amount']
desc = df_clean[key_vars].agg(['mean', 'median', 'std', 'min', 'max'])
print('--- Descriptive Statistics: Key Continuous Variables ---')
print(desc.round(2).to_string())

### 2.1 Descriptive Statistics — Categorical Variables (All Categories)

In [ ]:
# Count and percentage for ALL categories of each categorical variable
for cat in ['grade', 'verified_income', 'homeownership']:
    vc  = df_clean[cat].value_counts().sort_index()
    pct = (vc / len(df_clean) * 100).round(2)
    print(f'\n{cat.upper()}:')
    print(f'  {"Category":<20} {"Count":>8} {"Percentage":>12}')
    print(f'  {"-"*44}')
    for k in vc.index:
        print(f'  {str(k):<20} {vc[k]:>8,} {pct[k]:>11.2f}%')

### 2.2(a) Histograms

In [ ]:
# Figure 1: Distribution of Interest Rate
fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(df_clean['interest_rate'].dropna(), bins=40,
        color='steelblue', edgecolor='white')
ax.set_title('Distribution of Interest Rate', fontsize=14, fontweight='bold')
ax.set_xlabel('Interest Rate (%)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
plt.tight_layout()
plt.savefig('fig1_hist_interest_rate.png', dpi=150)
plt.show()
print('Figure 1 saved: fig1_hist_interest_rate.png')

In [ ]:
# Figure 2: Distribution of Annual Income
fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(df_clean['annual_income'].dropna(), bins=60,
        color='darkorange', edgecolor='white')
ax.set_title('Distribution of Annual Income', fontsize=14, fontweight='bold')
ax.set_xlabel('Annual Income (USD)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
plt.tight_layout()
plt.savefig('fig2_hist_annual_income.png', dpi=150)
plt.show()
print('Figure 2 saved: fig2_hist_annual_income.png')

### 2.2(b) Scatterplots

In [ ]:
# Figure 3: Interest Rate vs Annual Income
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df_clean['annual_income'], df_clean['interest_rate'],
           alpha=0.15, color='steelblue', s=8)
ax.set_title('Interest Rate vs Annual Income', fontsize=14, fontweight='bold')
ax.set_xlabel('Annual Income (USD)', fontsize=12)
ax.set_ylabel('Interest Rate (%)', fontsize=12)
plt.tight_layout()
plt.savefig('fig3_scatter_income.png', dpi=150)
plt.show()
print('Figure 3 saved: fig3_scatter_income.png')

In [ ]:
# Figure 4: Interest Rate vs DTI with OLS regression line
sub = df_clean[['interest_rate', 'debt_to_income']].dropna()
slope, intercept, r_val, p_val, se_slope = stats.linregress(
    sub['debt_to_income'], sub['interest_rate'])
x_range = np.linspace(sub['debt_to_income'].min(),
                      sub['debt_to_income'].max(), 200)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(sub['debt_to_income'], sub['interest_rate'],
           alpha=0.15, color='darkorange', s=8)
ax.plot(x_range, intercept + slope * x_range,
        color='navy', linewidth=2, label=f'OLS slope = {slope:.4f}')
ax.set_title('Interest Rate vs Debt-to-Income (with OLS Regression Line)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Debt-to-Income Ratio', fontsize=12)
ax.set_ylabel('Interest Rate (%)', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('fig4_scatter_dti.png', dpi=150)
plt.show()
print('Figure 4 saved: fig4_scatter_dti.png')

### 2.2(c) Boxplots

In [ ]:
# Figure 5: Interest Rate by Loan Grade
fig, ax = plt.subplots(figsize=(8, 6))
df_clean.boxplot(column='interest_rate', by='grade', ax=ax, grid=False,
                 boxprops=dict(color='steelblue'),
                 medianprops=dict(color='red', linewidth=2))
ax.set_title('Interest Rate by Loan Grade', fontsize=14, fontweight='bold')
ax.set_xlabel('Grade', fontsize=12)
ax.set_ylabel('Interest Rate (%)', fontsize=12)
plt.suptitle('')   # Remove auto-generated pandas title
plt.tight_layout()
plt.savefig('fig5_box_grade.png', dpi=150)
plt.show()
print('Figure 5 saved: fig5_box_grade.png')

In [ ]:
# Figure 6: Interest Rate by Income Verification Status
fig, ax = plt.subplots(figsize=(8, 6))
df_clean.boxplot(column='interest_rate', by='verified_income', ax=ax, grid=False,
                 boxprops=dict(color='steelblue'),
                 medianprops=dict(color='red', linewidth=2))
ax.set_title('Interest Rate by Income Verification Status',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Verified Income', fontsize=12)
ax.set_ylabel('Interest Rate (%)', fontsize=12)
plt.suptitle('')
plt.tight_layout()
plt.savefig('fig6_box_verified_income.png', dpi=150)
plt.show()
print('Figure 6 saved: fig6_box_verified_income.png')

In [ ]:
# Figure 7: Interest Rate by Homeownership Status
fig, ax = plt.subplots(figsize=(8, 6))
df_clean.boxplot(column='interest_rate', by='homeownership', ax=ax, grid=False,
                 boxprops=dict(color='steelblue'),
                 medianprops=dict(color='red', linewidth=2))
ax.set_title('Interest Rate by Homeownership Status',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Homeownership', fontsize=12)
ax.set_ylabel('Interest Rate (%)', fontsize=12)
plt.suptitle('')
plt.tight_layout()
plt.savefig('fig7_box_homeownership.png', dpi=150)
plt.show()
print('Figure 7 saved: fig7_box_homeownership.png')

### 2.3 Feature Engineering

In [ ]:
# (a) Credit Utilization Ratio
# Formula: total_credit_utilized / total_credit_limit
# Assign 0 where total_credit_limit = 0 to avoid division by zero errors
df_clean['credit_util'] = np.where(
    df_clean['total_credit_limit'] == 0,
    0,
    df_clean['total_credit_utilized'] / df_clean['total_credit_limit']
)

cu_mean    = df_clean['credit_util'].mean()
cu_nonzero = (df_clean['credit_util'] != 0).mean() * 100
print(f'credit_util    — Mean: {cu_mean:.4f}   % Non-zero: {cu_nonzero:.2f}%')

# (b) Bankruptcy Indicator
# 1 if public_record_bankrupt >= 1 (at least one bankruptcy), else 0
df_clean['bankruptcy_dummy'] = (
    df_clean['public_record_bankrupt'] >= 1).astype(int)

bk_mean    = df_clean['bankruptcy_dummy'].mean()
bk_nonzero = (df_clean['bankruptcy_dummy'] != 0).mean() * 100
print(f'bankruptcy_dummy — Mean: {bk_mean:.4f}   % Non-zero: {bk_nonzero:.2f}%')

---
# Part C: Regression Analysis

### Prepare Regression Sample

In [ ]:
# Drop rows missing any variable used in regression (listwise deletion)
df_reg = df_clean.dropna(subset=[
    'interest_rate', 'debt_to_income', 'credit_util', 'bankruptcy_dummy',
    'annual_income', 'loan_amount', 'term', 'grade', 'emp_length',
    'homeownership', 'loan_purpose', 'credit_checks'
]).copy()

print(f'Full sample n     : {len(df_clean)}')
print(f'Regression sample : {len(df_reg)} (dropped {len(df_clean)-len(df_reg)} rows with missing values)')

# Create verified_income dummies — reference category = Not Verified
df_reg['vi_source']   = (df_reg['verified_income'] == 'Source Verified').astype(float)
df_reg['vi_verified'] = (df_reg['verified_income'] == 'Verified').astype(float)
print('Dummy variables created for verified_income (ref = Not Verified)')

### 3.1(a) Model 1: Simple Linear Regression — Debt-to-Income

In [ ]:
m1 = smf.ols('interest_rate ~ debt_to_income', data=df_reg).fit()

print('MODEL 1: interest_rate ~ debt_to_income')
print('=' * 60)
print(f'Fitted equation:')
print(f'  interest_rate = {m1.params["Intercept"]:.4f} + {m1.params["debt_to_income"]:.4f} x debt_to_income')
print(f'')
print(f'  Intercept  = {m1.params["Intercept"]:.4f}  (SE = {m1.bse["Intercept"]:.4f})')
print(f'  beta_1     = {m1.params["debt_to_income"]:.4f}  (SE = {m1.bse["debt_to_income"]:.4f})')
print(f'  p-value    = {m1.pvalues["debt_to_income"]:.2e}')
print(f'  R2         = {m1.rsquared:.4f}')
print(f'  F-stat     = {m1.fvalue:.2f}')
print(f'  N          = {int(m1.nobs)}')
print()
print(m1.summary())

### 3.1(b) Model 2: Simple Linear Regression — Bankruptcy Dummy

In [ ]:
m2 = smf.ols('interest_rate ~ bankruptcy_dummy', data=df_reg).fit()

print('MODEL 2: interest_rate ~ bankruptcy_dummy')
print('=' * 60)
print(f'Fitted equation:')
print(f'  interest_rate = {m2.params["Intercept"]:.4f} + {m2.params["bankruptcy_dummy"]:.4f} x bankruptcy_dummy')
print(f'')
print(f'  Intercept  = {m2.params["Intercept"]:.4f}  (SE = {m2.bse["Intercept"]:.4f})')
print(f'  beta_1     = {m2.params["bankruptcy_dummy"]:.4f}  (SE = {m2.bse["bankruptcy_dummy"]:.4f})')
print(f'  p-value    = {m2.pvalues["bankruptcy_dummy"]:.6f}')
print(f'  R2         = {m2.rsquared:.4f}')
print(f'  F-stat     = {m2.fvalue:.2f}')
print(f'  N          = {int(m2.nobs)}')
print()
print(m2.summary())

### 3.1(c) Model 3: Categorical Variable Regression — Verified Income
**Reference category: Not Verified**  
Two dummies created: `vi_source` (Source Verified) and `vi_verified` (Verified)

In [ ]:
m3 = smf.ols('interest_rate ~ vi_source + vi_verified', data=df_reg).fit()

print('MODEL 3: interest_rate ~ vi_source + vi_verified')
print('Reference category: Not Verified')
print('=' * 60)
print(f'Fitted equation:')
print(f'  interest_rate = {m3.params["Intercept"]:.4f} + {m3.params["vi_source"]:.4f} x D_source + {m3.params["vi_verified"]:.4f} x D_verified')
print(f'')
print(f'  Intercept (Not Verified predicted rate) = {m3.params["Intercept"]:.4f}%')
print(f'  beta_1 vi_source   = {m3.params["vi_source"]:.4f}  (SE={m3.bse["vi_source"]:.4f}, p={m3.pvalues["vi_source"]:.2e})')
print(f'  beta_2 vi_verified = {m3.params["vi_verified"]:.4f}  (SE={m3.bse["vi_verified"]:.4f}, p={m3.pvalues["vi_verified"]:.2e})')
print(f'  R2     = {m3.rsquared:.4f}')
print(f'  F-stat = {m3.fvalue:.2f}')
print()
print(m3.summary())

### 3.1(d) Model 4: Multiple Regression

In [ ]:
m4 = smf.ols(
    'interest_rate ~ debt_to_income + credit_util + bankruptcy_dummy',
    data=df_reg).fit()

print('MODEL 4: interest_rate ~ debt_to_income + credit_util + bankruptcy_dummy')
print('=' * 60)
print(f'Fitted equation:')
print(f'  interest_rate = {m4.params["Intercept"]:.4f}')
print(f'                + {m4.params["debt_to_income"]:.4f} x debt_to_income')
print(f'                + {m4.params["credit_util"]:.4f} x credit_util')
print(f'                + {m4.params["bankruptcy_dummy"]:.4f} x bankruptcy_dummy')
print()
for var in ['Intercept', 'debt_to_income', 'credit_util', 'bankruptcy_dummy']:
    print(f'  {var:<22} coef={m4.params[var]:>8.4f}  SE={m4.bse[var]:.4f}  p={m4.pvalues[var]:.2e}')
print(f'  R2     = {m4.rsquared:.4f}')
print(f'  F-stat = {m4.fvalue:.2f}')
print(f'  N      = {int(m4.nobs)}')
print()
# Show DTI coefficient change vs Model 1
print('--- DTI Coefficient Comparison (Omitted Variable Bias) ---')
print(f'  Model 1 DTI coef: {m1.params["debt_to_income"]:.4f}')
print(f'  Model 4 DTI coef: {m4.params["debt_to_income"]:.4f}')
print(f'  Change          : {m1.params["debt_to_income"]-m4.params["debt_to_income"]:.4f} pp (reduction due to omitted variable bias in M1)')
print()
print(m4.summary())

### 3.1(e) Model 5: Full Specification
**Reference categories:**
- `grade` = A
- `homeownership` = RENT
- `emp_length` = 10 years (most common)
- `loan_purpose` = debt_consolidation
- `verified_income` = Not Verified

In [ ]:
# Build the X matrix manually — gives full control over reference categories
# This avoids the issues with pd.get_dummies (wrong refs, includes issue_month etc.)

# Step 1: Start with continuous and pre-encoded variables
Xdf = df_reg[[
    'debt_to_income', 'credit_util', 'annual_income',
    'loan_amount', 'credit_checks', 'vi_source', 'vi_verified'
]].copy()
Xdf['bankruptcy_dummy'] = df_reg['bankruptcy_dummy'].astype(float)
Xdf['term_60'] = (df_reg['term'] == 60).astype(float)   # ref = 36 months

# Step 2: Grade dummies — reference category = A
for g in ['B', 'C', 'D', 'E', 'F', 'G']:
    Xdf[f'grade_{g}'] = (df_reg['grade'] == g).astype(float)

# Step 3: Homeownership dummies — reference category = RENT
for h in ['MORTGAGE', 'OWN']:
    Xdf[f'home_{h}'] = (df_reg['homeownership'] == h).astype(float)

# Step 4: Employment length dummies — reference = most common category (10 years)
emp_ref = df_reg['emp_length'].value_counts().index[0]
print(f'emp_length reference category: {emp_ref}')
for e in df_reg['emp_length'].unique():
    if e != emp_ref:
        safe = str(e).replace(' ', '_').replace('.', '_').replace('+', 'p')
        Xdf[f'emp_{safe}'] = (df_reg['emp_length'] == e).astype(float)

# Step 5: Loan purpose dummies — reference category = debt_consolidation
for p in df_reg['loan_purpose'].unique():
    if p != 'debt_consolidation':
        safe = str(p).replace(' ', '_').replace('-', '_')
        Xdf[f'pur_{safe}'] = (df_reg['loan_purpose'] == p).astype(float)

# Step 6: Fit Model 5 using sm.OLS with constant
y          = df_reg['interest_rate'].values
m5         = sm.OLS(y, sm.add_constant(Xdf.values)).fit()
col_names  = ['const'] + list(Xdf.columns)
coef5      = pd.Series(m5.params, index=col_names)
se5        = pd.Series(m5.bse,    index=col_names)
pval5      = pd.Series(m5.pvalues,index=col_names)

print(f'Model 5 fitted successfully')
print(f'  R2     = {m5.rsquared:.4f}')
print(f'  F-stat = {m5.fvalue:.0f}')
print(f'  N      = {int(m5.nobs)}')

In [ ]:
# Print key Model 5 coefficients with significance stars
def stars(p):
    return '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else '  '))

key_cols = [
    'const', 'debt_to_income', 'credit_util', 'bankruptcy_dummy',
    'annual_income', 'loan_amount', 'credit_checks', 'term_60',
    'vi_source', 'vi_verified',
    'grade_B', 'grade_C', 'grade_D', 'grade_E', 'grade_F', 'grade_G',
    'home_MORTGAGE', 'home_OWN'
]

print(f'MODEL 5 KEY COEFFICIENTS')
print(f'Reference: grade A | homeownership RENT | emp_length {emp_ref} yrs | loan_purpose debt_consolidation')
print('=' * 72)
print(f'{"Variable":<28} {"Coef":>10} {"SE":>10} {"p-value":>10}  Sig')
print('-' * 72)
for k in key_cols:
    if k in coef5.index:
        p = pval5[k]
        print(f'{k:<28} {coef5[k]:>10.4f} {se5[k]:>10.4f} {p:>10.4f}  {stars(p)}')

### Model 5 Residuals — First Five Observations

In [ ]:
# Extract residuals for the first five observations
# Note: m5 uses sm.OLS so fittedvalues is a numpy array (not pandas Series)
fitted5 = m5.fittedvalues   # numpy array
resid5  = y - fitted5       # numpy array

print('Residuals - First 5 Observations (Model 5)')
print('=' * 52)
# Use variables for header to avoid f-string parenthesis conflict
col1 = 'Actual (%)'
col2 = 'Predicted (%)'
col3 = 'Residual'
print(f'Row   {col1:>12} {col2:>14} {col3:>10}')
print('-' * 52)
for i in range(5):
    print(f'{i:<5} {y[i]:>12.2f} {fitted5[i]:>14.2f} {resid5[i]:>10.4f}')


### 3.2 Regression Summary Table — All Five Models

In [ ]:
# Format helper: coefficient with significance stars and SE in parentheses
def fmt_coef(model, var):
    if var not in model.params.index:
        return '—'
    c = model.params[var]
    s = model.bse[var]
    p = model.pvalues[var]
    st = '***' if p<0.01 else ('**' if p<0.05 else ('*' if p<0.10 else ''))
    return f'{c:.4f}{st} ({s:.4f})'

def fmt_m5(key):
    if key not in coef5.index:
        return '—'
    c = coef5[key]; s = se5[key]; p = pval5[key]
    st = '***' if p<0.01 else ('**' if p<0.05 else ('*' if p<0.10 else ''))
    return f'{c:.4f}{st} ({s:.4f})'

# Print table
print(f'{"Variable":<28} {"Model 1":>22} {"Model 2":>22} {"Model 3":>22} {"Model 4":>22} {"Model 5":>22}')
print('=' * 142)

rows = [
    ('Intercept',          'Intercept',       'Intercept',       'Intercept',       'Intercept',       'const'),
    ('debt_to_income',     'debt_to_income',  None,              None,              'debt_to_income',  'debt_to_income'),
    ('bankruptcy_dummy',   None,              'bankruptcy_dummy',None,              'bankruptcy_dummy','bankruptcy_dummy'),
    ('credit_util',        None,              None,              None,              'credit_util',     'credit_util'),
    ('vi_source',          None,              None,              'vi_source',       None,              'vi_source'),
    ('vi_verified',        None,              None,              'vi_verified',     None,              'vi_verified'),
    ('grade_B (ref=A)',    None,              None,              None,              None,              'grade_B'),
    ('grade_C',            None,              None,              None,              None,              'grade_C'),
    ('grade_D',            None,              None,              None,              None,              'grade_D'),
    ('grade_E',            None,              None,              None,              None,              'grade_E'),
    ('grade_F',            None,              None,              None,              None,              'grade_F'),
    ('grade_G',            None,              None,              None,              None,              'grade_G'),
    ('home_MORTGAGE',      None,              None,              None,              None,              'home_MORTGAGE'),
    ('home_OWN',           None,              None,              None,              None,              'home_OWN'),
    ('annual_income',      None,              None,              None,              None,              'annual_income'),
    ('loan_amount',        None,              None,              None,              None,              'loan_amount'),
    ('credit_checks',      None,              None,              None,              None,              'credit_checks'),
    ('term_60',            None,              None,              None,              None,              'term_60'),
    ('emp_length dummies', None,              None,              None,              None,              'INCLUDED'),
    ('loan_purpose dummies',None,             None,              None,              None,              'INCLUDED'),
]

for label, v1, v2, v3, v4, v5 in rows:
    c1 = fmt_coef(m1, v1) if v1 else '—'
    c2 = fmt_coef(m2, v2) if v2 else '—'
    c3 = fmt_coef(m3, v3) if v3 else '—'
    c4 = fmt_coef(m4, v4) if v4 else '—'
    c5 = fmt_m5(v5)       if v5 and v5 != 'INCLUDED' else ('Included' if v5 == 'INCLUDED' else '—')
    print(f'{label:<28} {c1:>22} {c2:>22} {c3:>22} {c4:>22} {c5:>22}')

print('=' * 142)
print(f'{"R2":<28} {m1.rsquared:>22.4f} {m2.rsquared:>22.4f} {m3.rsquared:>22.4f} {m4.rsquared:>22.4f} {m5.rsquared:>22.4f}')
print(f'{"N":<28} {int(m1.nobs):>22,} {int(m2.nobs):>22,} {int(m3.nobs):>22,} {int(m4.nobs):>22,} {int(m5.nobs):>22,}')
print(f'{"F-statistic":<28} {m1.fvalue:>22.1f} {m2.fvalue:>22.1f} {m3.fvalue:>22.1f} {m4.fvalue:>22.1f} {m5.fvalue:>22.0f}')
print()
print('* p<0.10  ** p<0.05  *** p<0.01')
print('Standard errors in parentheses.')
print('Reference categories — grade: A | homeownership: RENT | emp_length: 10 yrs | loan_purpose: debt_consolidation')

---
## Submission Checklist

Before submitting, verify:

- [ ] This notebook runs **top to bottom without errors** (Kernel > Restart & Run All)
- [ ] `loans_dataset.csv` is in the **same folder** as this notebook
- [ ] All **7 figures** have been saved as PNG files in the working directory
- [ ] This notebook is uploaded to a **public GitHub repository**
- [ ] The **GitHub link** is included on the cover page of your PDF report

**Figures produced:** fig1 through fig7 (histograms, scatterplots, boxplots)